<a href="https://colab.research.google.com/github/Maryam-Yaqoob/NLP-LAB/blob/main/FA23_BAI_025___Lab_2____Maryam_Yaqoob.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 02 — Data Acquisition for NLP: Full Solutions
This notebook solves all nine tasks: PDF extraction & word frequency, structured PDF metadata + CSV export, a PDF preprocessing pipeline, DOCX paragraph classification, DOCX table extraction with pandas, multi-document comparison/deduplication, and three API-based analyses using `jsonplaceholder.typicode.com`.

**Files used** (place these in the same folder as this notebook, or point the paths at your own files):
- `sample_multipage.pdf` — a 6-page sample PDF (Tasks 1-3)
- `document_A.docx` — the provided "Data Acquisition for NLP" document (Tasks 4 & 6)
- `document_B.docx` — a second sample document paired with A (Task 6)
- `table_doc.docx` — a sample document containing one table (Task 5)

**Note on `input()` cells:** Tasks 1(d) and 2(c) ask for keyboard input. Those cells call Python's real `input()` — run them yourself and type a value when prompted; they are left un-executed in the saved notebook since they need an interactive session.

**Note on Tasks 7-9:** these need live internet access to `jsonplaceholder.typicode.com`. The code is correct and will run as-is on any machine with a normal internet connection.

In [18]:
# Setup: install & import everything needed for the whole notebook
import sys, subprocess

def _pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# Uncomment on first run in a fresh environment:
for pkg in ["PyPDF2", "pdfplumber", "python-docx", "pandas", "requests", "nltk"]:
    _pip_install(pkg)

import re
import csv
import math
import json
from collections import Counter

import PyPDF2
import docx
import pandas as pd
import requests
import nltk

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")
from nltk.corpus import stopwords as nltk_stopwords

PDF_PATH = "/content/sample_multipage.pdf"
DOCX_A_PATH = "/content/document_A.docx"
DOCX_B_PATH = "d/content/document_B.docx"
TABLE_DOCX_PATH = "/content/table_doc.docx"

print("Setup complete.")


Setup complete.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


---
# Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

**(a) Extract text from each page individually into a list of strings; print total page count**

In [19]:
with open(PDF_PATH, "rb") as f:
    reader = PyPDF2.PdfReader(f)
    pages_list = [page.extract_text() or "" for page in reader.pages]

print("Total number of pages extracted:", len(pages_list))
print("\n--- Page 1 preview ---\n", pages_list[0][:150])


Total number of pages extracted: 6

--- Page 1 preview ---
 Natural Language Processing Overview
Natural language processing, often shortened to NLP, is a field of artificial
intelligence that gives machines th


**(b) Merge pages, strip punctuation/digits, lowercase, tokenize on whitespace; print token count**

In [20]:
full_text = " ".join(pages_list)
cleaned_for_tokens = re.sub(r"[^a-zA-Z\s]", "", full_text).lower()
tokens = cleaned_for_tokens.split()

print("Total token count:", len(tokens))
print("First 15 tokens:", tokens[:15])


Total token count: 302
First 15 tokens: ['natural', 'language', 'processing', 'overview', 'natural', 'language', 'processing', 'often', 'shortened', 'to', 'nlp', 'is', 'a', 'field', 'of']


**(c) Word frequency dictionary with stopwords removed; top 10 most frequent words**

In [21]:
stopwords_20 = {
    "the", "is", "at", "which", "on", "and", "a", "an", "of", "to",
    "in", "for", "with", "that", "this", "as", "are", "be", "or", "from"
}

filtered_tokens = [t for t in tokens if t not in stopwords_20]
freq = Counter(filtered_tokens)

print("Top 10 most frequent words:")
for word, count in freq.most_common(10):
    print(f"  {word:<15} {count}")


Top 10 most frequent words:
  data            9
  python          6
  nlp             5
  such            5
  tokenization    5
  word            4
  frequency       4
  language        3
  libraries       3
  acquisition     3


**(d) Search a keyword across pages, print every matching sentence with its page number**
*(interactive — run this cell yourself and type a keyword when prompted)*

In [22]:
try:
    keyword = input("Enter a keyword to search: ").strip().lower()
except Exception:
    keyword = "python"  # fallback so the cell can still run non-interactively
    print(f"(No interactive input available — using default keyword '{keyword}')")

found_any = False
for page_num, page_text in enumerate(pages_list, start=1):
    sentences = re.split(r"(?<=[.!?])\s+", page_text.strip())
    for sentence in sentences:
        if keyword and keyword in sentence.lower():
            found_any = True
            print(f"[Page {page_num}] {sentence.strip()}")

if not found_any:
    print(f"No sentences containing '{keyword}' were found.")


Enter a keyword to search: learning
No sentences containing 'learning' were found.


---
# Task 2: Structured Page Metadata Extraction and CSV Export

**(a) Build `pages_data`: one dict per page with page_number, raw_text, word_count, char_count**

In [23]:
pages_data = []
for i, page_text in enumerate(pages_list, start=1):
    words = page_text.split()
    char_count = len(page_text.replace(" ", "").replace("\n", ""))
    pages_data.append({
        "page_number": i,
        "raw_text": page_text,
        "word_count": len(words),
        "char_count": char_count,
    })

print("First 2 entries of pages_data:")
for entry in pages_data[:2]:
    print(entry)


First 2 entries of pages_data:
{'page_number': 1, 'raw_text': 'Natural Language Processing Overview\nNatural language processing, often shortened to NLP, is a field of artificial\nintelligence that gives machines the ability to read, understand and derive\nmeaning from human languages. Python is the most widely used programming\nlanguage for NLP because of its rich ecosystem of libraries. You can learn\nmore at https://www.nlp-resources.example.com/intro for a general overview.\nNLP powers applications such as chatbots, translation engines and search.\n', 'word_count': 68, 'char_count': 421}
{'page_number': 2, 'raw_text': 'Data Acquisition Challenges\nData acquisition is the process of gathering and preparing data for an NLP\nproject in the year 2023. There are 3 broad categories of data acquisition:\navailable data, other sources of data such as APIs and PDFs, and situations\nwhere no data is available at all. Python developers often rely on libraries\nsuch as requests and BeautifulSo

**(b) Highest word-count page, lowest non-zero word-count page, average word count**

In [24]:
nonzero_pages = [p for p in pages_data if p["word_count"] > 0]

highest_page = max(pages_data, key=lambda p: p["word_count"])
lowest_nonzero_page = min(nonzero_pages, key=lambda p: p["word_count"]) if nonzero_pages else None
average_word_count = round(
    sum(p["word_count"] for p in pages_data) / len(pages_data), 2
)

print(f"Highest word count -> Page {highest_page['page_number']} ({highest_page['word_count']} words)")
print(highest_page["raw_text"])
print()
if lowest_nonzero_page:
    print(f"Lowest non-zero word count -> Page {lowest_nonzero_page['page_number']} "
          f"({lowest_nonzero_page['word_count']} words)")
else:
    print("No pages with a non-zero word count were found.")
print(f"Average word count across all pages: {average_word_count}")

zero_word_pages = [p["page_number"] for p in pages_data if p["word_count"] == 0]
print("Pages with zero words:", zero_word_pages)


Highest word count -> Page 1 (68 words)
Natural Language Processing Overview
Natural language processing, often shortened to NLP, is a field of artificial
intelligence that gives machines the ability to read, understand and derive
meaning from human languages. Python is the most widely used programming
language for NLP because of its rich ecosystem of libraries. You can learn
more at https://www.nlp-resources.example.com/intro for a general overview.
NLP powers applications such as chatbots, translation engines and search.


Lowest non-zero word count -> Page 3 (54 words)
Average word count across all pages: 51.67
Pages with zero words: [6]


**(c) Case-insensitive phrase search across pages; custom `PhraseNotFoundError`**
*(interactive — run this cell yourself and type a phrase when prompted)*

In [25]:
class PhraseNotFoundError(Exception):
    """Raised when a search phrase is not found on any page."""
    pass

try:
    search_phrase = input("Enter a search phrase: ").strip().lower()
except Exception:
    search_phrase = "data acquisition"  # fallback so the cell can still run non-interactively
    print(f"(No interactive input available — using default phrase '{search_phrase}')")

try:
    matches = {}
    for p in pages_data:
        count = p["raw_text"].lower().count(search_phrase)
        if count > 0:
            matches[p["page_number"]] = count

    if not matches:
        raise PhraseNotFoundError(f"The phrase '{search_phrase}' was not found on any page.")

    for page_number, count in matches.items():
        print(f"Page {page_number}: '{search_phrase}' appears {count} time(s)")

except PhraseNotFoundError as e:
    print("Search failed:", e)


Enter a search phrase: reinforcement learning
Search failed: The phrase 'reinforcement learning' was not found on any page.


**(d) Write `pdf_summary.csv` with `page_number, word_count, char_count`; re-read and print rows**

In [26]:
csv_path = "pdf_summary.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for p in pages_data:
        writer.writerow([p["page_number"], p["word_count"], p["char_count"]])

print(f"Wrote {csv_path}\n")

with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)


Wrote pdf_summary.csv

['page_number', 'word_count', 'char_count']
['1', '68', '421']
['2', '68', '337']
['3', '54', '384']
['4', '63', '329']
['5', '57', '352']
['6', '0', '0']


---
# Task 3: PDF Text Preprocessing Pipeline for NLP

**(a) Extract all text; clean it (remove URLs, digits, non-alpha chars, collapse spaces)**

In [27]:
with open(PDF_PATH, "rb") as f:
    reader3 = PyPDF2.PdfReader(f)
    raw_text_all = " ".join(page.extract_text() or "" for page in reader3.pages)

char_count_before = len(raw_text_all)

step1 = re.sub(r"https?://\S+|www\.\S+", " ", raw_text_all)      # (i) remove URLs
step2 = re.sub(r"\d+", " ", step1)                                # (ii) remove digits
step3 = re.sub(r"[^a-zA-Z\s.]", " ", step2)                        # (iii) keep letters, spaces, full stops
cleaned_text = re.sub(r"\s+", " ", step3).strip()                  # (iv) collapse spaces

char_count_after = len(cleaned_text)

print("Character count before cleaning:", char_count_before)
print("Character count after cleaning:", char_count_after)


Character count before cleaning: 2142
Character count after cleaning: 1980


**(b) Lowercase, split into sentences with `re.split(r'[.!?]', text)`, strip, drop empties**

In [28]:
lowered = cleaned_text.lower()
raw_sentences = re.split(r"[.!?]", lowered)
sentence_count_before = len(raw_sentences)

sentences = [s.strip() for s in raw_sentences if s.strip()]
sentence_count_after = len(sentences)

print("Sentence count before removing empty strings:", sentence_count_before)
print("Sentence count after removing empty strings:", sentence_count_after)


Sentence count before removing empty strings: 23
Sentence count after removing empty strings: 22


**(c) Tokenize sentences, remove NLTK stopwords, discard sentences with <5 tokens left**

In [29]:
stop_words_nltk = set(nltk_stopwords.words("english"))

sentence_count_before_filter = len(sentences)
filtered_sentences = []
for sent in sentences:
    tokens_s = sent.split()
    tokens_no_stop = [t for t in tokens_s if t not in stop_words_nltk]
    if len(tokens_no_stop) >= 5:
        filtered_sentences.append(sent)

sentence_count_after_filter = len(filtered_sentences)

print("Sentence count before stopword-length filter:", sentence_count_before_filter)
print("Sentence count after stopword-length filter:", sentence_count_after_filter)
print("\n3 sample sentences:")
for s in filtered_sentences[:3]:
    print("-", s)


Sentence count before stopword-length filter: 22
Sentence count after stopword-length filter: 18

3 sample sentences:
- natural language processing overview natural language processing often shortened to nlp is a field of artificial intelligence that gives machines the ability to read understand and derive meaning from human languages
- python is the most widely used programming language for nlp because of its rich ecosystem of libraries
- nlp powers applications such as chatbots translation engines and search


**(d) Write remaining sentences to `nlp_ready_output.txt`, re-read, report line/length stats**

In [30]:
out_txt_path = "nlp_ready_output.txt"

with open(out_txt_path, "w", encoding="utf-8") as f:
    for s in filtered_sentences:
        f.write(s + "\n")

with open(out_txt_path, "r", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f]

total_lines = len(lines)
longest_sentence = max(lines, key=len) if lines else ""
shortest_sentence = min(lines, key=len) if lines else ""

print("Total lines written:", total_lines)
print("Longest sentence:", longest_sentence)
print("Shortest sentence:", shortest_sentence)


Total lines written: 18
Longest sentence: natural language processing overview natural language processing often shortened to nlp is a field of artificial intelligence that gives machines the ability to read understand and derive meaning from human languages
Shortest sentence: khan both agree that clean data e


---
# Task 4: DOCX Paragraph Classification and Statistical Analysis

**(a) Store text + style name per paragraph; print total count and unique style names**

In [31]:
doc_a = docx.Document(DOCX_A_PATH)

paragraphs_a = [{"text": p.text, "style": p.style.name} for p in doc_a.paragraphs]

unique_styles = sorted({p["style"] for p in paragraphs_a})

print("Total paragraph count:", len(paragraphs_a))
print("Unique style names found:", unique_styles)


Total paragraph count: 41
Unique style names found: ['Heading 2', 'List Paragraph', 'Normal', 'pw-post-body-paragraph']


**(b) Classify paragraphs into Headings / Body / Other-Empty; print counts**

In [32]:
def classify_paragraph(p):
    style = p["style"]
    text = p["text"].strip()
    if style.startswith("Heading"):
        return "Headings"
    if style == "Normal" or "Body" in style:
        if text == "":
            return "Other/Empty"
        return "Body"
    return "Other/Empty"

groups = {"Headings": [], "Body": [], "Other/Empty": []}
for p in paragraphs_a:
    groups[classify_paragraph(p)].append(p)

for group_name, items in groups.items():
    print(f"{group_name}: {len(items)}")


Headings: 6
Body: 19
Other/Empty: 16


**(c) Body paragraphs with >5 words: highest/lowest word count, mean word count**

In [33]:
qualifying = []
for p in groups["Body"]:
    word_count = len(p["text"].split())
    char_count_no_spaces = len(p["text"].replace(" ", ""))
    if word_count > 5:
        qualifying.append({**p, "word_count": word_count, "char_count": char_count_no_spaces})

if qualifying:
    highest_wc = max(qualifying, key=lambda p: p["word_count"])
    lowest_wc = min(qualifying, key=lambda p: p["word_count"])
    mean_wc = round(sum(p["word_count"] for p in qualifying) / len(qualifying), 2)

    print("Highest word count body paragraph "
          f"({highest_wc['word_count']} words):\n  {highest_wc['text']}\n")
    print("Lowest word count (>5) body paragraph "
          f"({lowest_wc['word_count']} words):\n  {lowest_wc['text']}\n")
    print("Mean word count across qualifying body paragraphs:", mean_wc)
else:
    print("No body paragraphs with more than 5 words were found.")


Highest word count body paragraph (59 words):
  Transfer learning is a popular technique used in NLP that involves using pre-trained models to solve new tasks. Pre-trained models like BERT and GPT-2 have been trained on massive amounts of data and can be fine-tuned to solve new tasks with limited data. By leveraging transfer learning, we can save time and resources required to train models from scratch.

Lowest word count (>5) body paragraph (12 words):
  Audio: Speech-to-text tools can be used to convert audio into machine-readable text.

Mean word count across qualifying body paragraphs: 28.89


**(d) Nested dict `{style_name: {"count": N, "texts": [...]}}`, sorted by count desc, capped at 2 texts**

In [34]:
style_groups = {}
for p in paragraphs_a:
    entry = style_groups.setdefault(p["style"], {"count": 0, "texts": []})
    entry["count"] += 1
    entry["texts"].append(p["text"])

for entry in style_groups.values():
    if entry["count"] > 2:
        entry["texts"] = entry["texts"][:2]

sorted_style_groups = dict(
    sorted(style_groups.items(), key=lambda kv: kv[1]["count"], reverse=True)
)

import pprint
pprint.pprint(sorted_style_groups)


{'Heading 2': {'count': 6,
               'texts': ['Introduction to Data Acquisition for NLP',
                         'Three Types of Data Acquisition and Their '
                         'Challenges']},
 'List Paragraph': {'count': 3,
                    'texts': ['Data Generation:', 'Transfer Learning:']},
 'Normal': {'count': 27,
            'texts': ['Data Acquisition for NLP',
                      'Natural Language Processing (NLP) is a rapidly evolving '
                      'field that enables machines to understand and analyze '
                      'human language. One of the most crucial aspects of '
                      'building an effective NLP application is acquiring '
                      'high-quality data. However, data acquisition for NLP is '
                      'not always straightforward, and there are several '
                      'challenges involved in the process.']},
 'pw-post-body-paragraph': {'count': 5,
                            'texts': ['Da

---
# Task 5: DOCX Table Extraction and Pandas Integration

**(a) Extract every table as a list of lists; print table count and dimensions**

In [35]:
doc_t = docx.Document(TABLE_DOCX_PATH)

tables_data = []
for table in doc_t.tables:
    table_rows = [[cell.text for cell in row.cells] for row in table.rows]
    tables_data.append(table_rows)

print("Total number of tables found:", len(tables_data))
for i, t in enumerate(tables_data, start=1):
    n_rows = len(t)
    n_cols = len(t[0]) if n_rows else 0
    print(f"Table {i}: {n_rows} rows x {n_cols} columns")


Total number of tables found: 1
Table 1: 5 rows x 3 columns


**(b) First table -> pandas DataFrame (first row as header); print it, count empty/None cells per column**

In [36]:
first_table = tables_data[0]
header_row, *data_rows = first_table

df_table = pd.DataFrame(data_rows, columns=header_row)
df_table_display = df_table.replace("", pd.NA)

print(df_table)
print()
print("Empty/None cells per column:")
print(df_table_display.isna().sum())


          Name              Focus Area Years Active
0  Ayesha Khan         Computer Vision            2
1  Bilal Ahmed                                    1
2   Sara Malik                     NLP             
3  Usman Tariq  Reinforcement Learning            3

Empty/None cells per column:
Name            0
Focus Area      1
Years Active    1
dtype: int64


**(c) Paragraphs appearing before the first table (stop once a `w:tbl` element is hit)**

In [37]:
from docx.table import Table as DocxTable
from docx.text.paragraph import Paragraph as DocxParagraph

pre_table_paragraphs = []
for child in doc_t.element.body:
    if child.tag.endswith("}tbl"):
        break
    if child.tag.endswith("}p"):
        para = DocxParagraph(child, doc_t)
        pre_table_paragraphs.append(para.text)

print("Number of paragraphs before the first table:", len(pre_table_paragraphs))
for text in pre_table_paragraphs:
    print("-", text)


Number of paragraphs before the first table: 3
- Research Circle — Member Roster
- The table below lists a few AI/ML Co-Leads and members of the Research Circle along with their focus area and years active.
- Some cells are intentionally left blank to demonstrate handling of missing values when the table is loaded into a pandas DataFrame.


**(d) Write pre-table paragraphs and first table data into `docx_extracted.txt`**

In [38]:
extract_txt_path = "docx_extracted.txt"

with open(extract_txt_path, "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for text in pre_table_paragraphs:
        f.write(text + "\n")

    f.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        f.write(" | ".join(row) + "\n")

with open(extract_txt_path, "r", encoding="utf-8") as f:
    print(f.read())


--- PRE-TABLE PARAGRAPHS ---
Research Circle — Member Roster
The table below lists a few AI/ML Co-Leads and members of the Research Circle along with their focus area and years active.
Some cells are intentionally left blank to demonstrate handling of missing values when the table is loaded into a pandas DataFrame.

--- TABLE DATA ---
Name | Focus Area | Years Active
Ayesha Khan | Computer Vision | 2
Bilal Ahmed |  | 1
Sara Malik | NLP | 
Usman Tariq | Reinforcement Learning | 3



---
# Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

**(a) Headings from Document A and Document B; common / A-only / B-only via set operations**

In [41]:
doc_b = docx.Document("/content/document_B.docx")

headings_a = [p.text.strip() for p in doc_a.paragraphs if p.style.name.startswith("Heading")]
headings_b = [p.text.strip() for p in doc_b.paragraphs if p.style.name.startswith("Heading")]

set_a, set_b = set(headings_a), set(headings_b)

common_headings = set_a & set_b
only_in_a = set_a - set_b
only_in_b = set_b - set_a

print("Common headings:", common_headings)
print("Only in Document A:", only_in_a)
print("Only in Document B:", only_in_b)


Common headings: {'Conclusion:', 'Introduction to Data Acquisition for NLP', '2. Other Sources of Data: Public Datasets, Web Scraping, APIs, PDFs, Images, and Audio'}
Only in Document A: {'3. No Data Available: Overcoming the Challenge of Limited Data', 'Three Types of Data Acquisition and Their Challenges', '1. Available Data: Techniques for Data Retrieval and Augmentation'}
Only in Document B: {'Text Cleaning and Normalization Techniques', 'Stopword Removal and Lemmatization'}


**(b) "Normal" paragraphs from both docs, merged and deduplicated (case/whitespace-insensitive)**

In [42]:
normal_a = [p.text for p in doc_a.paragraphs if p.style.name == "Normal" and p.text.strip()]
normal_b = [p.text for p in doc_b.paragraphs if p.style.name == "Normal" and p.text.strip()]

combined_normal = normal_a + normal_b
count_before_dedup = len(combined_normal)

seen = set()
deduplicated_paragraphs = []
for text in combined_normal:
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        deduplicated_paragraphs.append(text)

count_after_dedup = len(deduplicated_paragraphs)

print("Total Normal paragraphs before deduplication:", count_before_dedup)
print("Total Normal paragraphs after deduplication:", count_after_dedup)


Total Normal paragraphs before deduplication: 28
Total Normal paragraphs after deduplication: 27


**(c) Tokenize deduplicated paragraphs, build a `Counter` word-frequency dict, remove stopwords, top 20**

In [43]:
stopwords_15 = {
    "the", "is", "at", "which", "on", "and", "a", "an", "of", "to",
    "in", "for", "with", "that", "this"
}

all_tokens_combined = []
for text in deduplicated_paragraphs:
    cleaned = re.sub(r"[^a-zA-Z\s]", "", text).lower()
    all_tokens_combined.extend(cleaned.split())

filtered_combined = [t for t in all_tokens_combined if t not in stopwords_15]
combined_freq = Counter(filtered_combined)

print("Top 20 most frequent words:")
for word, count in combined_freq.most_common(20):
    print(f"  {word:<15} {count}")


Top 20 most frequent words:
  data            37
  can             17
  nlp             13
  be              12
  available       9
  text            9
  using           8
  from            7
  as              7
  by              7
  or              6
  sources         6
  most            5
  techniques      5
  limited         5
  public          5
  such            5
  web             5
  apis            5
  involves        5


**(d) Summary: total headings, unique body paragraphs, vocabulary size, avg paragraph length, longest paragraph**

In [44]:
total_headings = len(headings_a) + len(headings_b)
total_unique_body_paragraphs = len(deduplicated_paragraphs)
vocabulary_size = len(combined_freq)

paragraph_word_counts = [len(text.split()) for text in deduplicated_paragraphs]
average_paragraph_length = round(
    sum(paragraph_word_counts) / len(paragraph_word_counts), 2
) if paragraph_word_counts else 0.0

longest_paragraph = max(deduplicated_paragraphs, key=lambda t: len(t.split())) if deduplicated_paragraphs else ""

print("Total headings across both documents:", total_headings)
print("Total unique body paragraphs after deduplication:", total_unique_body_paragraphs)
print("Total unique word types (vocabulary size):", vocabulary_size)
print("Average paragraph length in words:", average_paragraph_length)
print("Longest paragraph by word count:")
print(" ", longest_paragraph)


Total headings across both documents: 11
Total unique body paragraphs after deduplication: 27
Total unique word types (vocabulary size): 267
Average paragraph length in words: 25.33
Longest paragraph by word count:
  Transfer learning is a popular technique used in NLP that involves using pre-trained models to solve new tasks. Pre-trained models like BERT and GPT-2 have been trained on massive amounts of data and can be fine-tuned to solve new tasks with limited data. By leveraging transfer learning, we can save time and resources required to train models from scratch.


---
# Task 7: Multi-Endpoint API Data Joining and Analysis
*(requires live internet access to `jsonplaceholder.typicode.com`)*

**(a) GET both endpoints, verify status 200 (else raise `ConnectionError`), parse JSON**

In [45]:
USERS_URL = "https://jsonplaceholder.typicode.com/users"
POSTS_URL = "https://jsonplaceholder.typicode.com/posts"

def fetch_json(url):
    response = requests.get(url, timeout=10)
    if response.status_code != 200:
        raise ConnectionError(f"Request to {url} failed with status code {response.status_code}")
    return response.json()

try:
    users = fetch_json(USERS_URL)
    posts = fetch_json(POSTS_URL)
    print("Users fetched:", len(users))
    print("Posts fetched:", len(posts))
except Exception as e:
    # This environment has no internet access to jsonplaceholder.typicode.com,
    # so we fall back to empty lists purely so the rest of the notebook can be
    # inspected without crashing. On a machine with normal internet access,
    # fetch_json() above works exactly as specified and this except branch
    # never runs.
    print("Could not reach the API from this environment:", e)
    users, posts = [], []


Users fetched: 10
Posts fetched: 100


**(b) Join `users` (`id`) with `posts` (`userId`); build the nested dictionary; print first 3 users**

In [46]:
joined = {}
for user in users:
    user_posts = [p for p in posts if p["userId"] == user["id"]]
    joined[user["name"]] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": [p["title"] for p in user_posts],
    }

for name in list(joined)[:3]:
    print(name, "->", joined[name])


Leanne Graham -> {'email': 'Sincere@april.biz', 'post_count': 10, 'titles': ['sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'qui est esse', 'ea molestias quasi exercitationem repellat qui ipsa sit aut', 'eum et est occaecati', 'nesciunt quas odio', 'dolorem eum magni eos aperiam quia', 'magnam facilis autem', 'dolorem dolore est ipsam', 'nesciunt iure omnis dolorem tempora et accusantium', 'optio molestias id quia eum']}
Ervin Howell -> {'email': 'Shanna@melissa.tv', 'post_count': 10, 'titles': ['et ea vero quia laudantium autem', 'in quibusdam tempore odit est dolorem', 'dolorum ut in voluptas mollitia et saepe quo animi', 'voluptatem eligendi optio', 'eveniet quod temporibus', 'sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio', 'fugit voluptas sed molestias voluptatem provident', 'voluptate et itaque vero tempora molestiae', 'adipisci placeat illum aut reiciendis qui', 'doloribus ad provident suscipit at']}
Clementine Bauch -> {'email

**(c) User with the most posts: full name, email, company name, numbered post titles**

In [47]:
if joined:
    top_user_name = max(joined, key=lambda n: joined[n]["post_count"])
    top_user_record = next(u for u in users if u["name"] == top_user_name)

    print("Full name:", top_user_name)
    print("Email:", joined[top_user_name]["email"])
    print("Company name:", top_user_record["company"]["name"])
    print("Post titles:")
    for i, title in enumerate(joined[top_user_name]["titles"], start=1):
        print(f"  {i}. {title}")
else:
    print("No data available (fetch the API with internet access to populate this).")


Full name: Leanne Graham
Email: Sincere@april.biz
Company name: Romaguera-Crona
Post titles:
  1. sunt aut facere repellat provident occaecati excepturi optio reprehenderit
  2. qui est esse
  3. ea molestias quasi exercitationem repellat qui ipsa sit aut
  4. eum et est occaecati
  5. nesciunt quas odio
  6. dolorem eum magni eos aperiam quia
  7. magnam facilis autem
  8. dolorem dolore est ipsam
  9. nesciunt iure omnis dolorem tempora et accusantium
  10. optio molestias id quia eum


**(d) Filter users where `company.bs` contains "synergies" OR `address.city` is longer than 8 characters**

In [48]:
matching_users = []
for user in users:
    bs_field = user["company"]["bs"].lower()
    city = user["address"]["city"]
    if "synergies" in bs_field or len(city) > 8:
        matching_users.append(user)

print(f"{'Name':<20}{'City':<20}{'Post Count'}")
for user in matching_users:
    post_count = joined.get(user["name"], {}).get("post_count", 0)
    print(f"{user['name']:<20}{user['address']['city']:<20}{post_count}")


Name                City                Post Count
Leanne Graham       Gwenborough         10
Ervin Howell        Wisokyburgh         10
Clementine Bauch    McKenziehaven       10
Patricia Lebsack    South Elvis         10
Chelsey Dietrich    Roscoeview          10
Mrs. Dennis SchulistSouth Christy       10
Kurtis Weissnat     Howemouth           10
Nicholas Runolfsdottir VAliyaview           10
Glenna Reichert     Bartholomebury      10
Clementina DuBuque  Lebsackbury         10


---
# Task 8: To-Do Completion Rate Analysis Across Users
*(requires live internet access to `jsonplaceholder.typicode.com`)*

**(a) Fetch and validate the todos endpoint; print total record count**

In [49]:
TODOS_URL = "https://jsonplaceholder.typicode.com/todos"

try:
    response = requests.get(TODOS_URL, timeout=10)
    if response.status_code != 200:
        raise ValueError(f"Unexpected status code: {response.status_code}")
    todos = response.json()
    if not isinstance(todos, list):
        raise ValueError("Response is not a list.")
    if len(todos) < 1:
        raise ValueError("Response list is empty.")
    print("Total todo records retrieved:", len(todos))
except Exception as e:
    print("Validation failed:", e, "\n(No internet access in this environment — falling back to an empty list.)")
    todos = []


Total todo records retrieved: 200


**(b) Group todos by `userId`; compute total/completed/incomplete; verify the counts add up**

In [50]:
todo_stats = {}
for todo in todos:
    uid = todo["userId"]
    stats = todo_stats.setdefault(uid, {"total": 0, "completed": 0, "incomplete": 0})
    stats["total"] += 1
    if todo["completed"]:
        stats["completed"] += 1
    else:
        stats["incomplete"] += 1

for uid in sorted(todo_stats):
    s = todo_stats[uid]
    checks_out = (s["completed"] + s["incomplete"]) == s["total"]
    print(f"User {uid}: {s}  ->  completed + incomplete == total: {checks_out}")


User 1: {'total': 20, 'completed': 11, 'incomplete': 9}  ->  completed + incomplete == total: True
User 2: {'total': 20, 'completed': 8, 'incomplete': 12}  ->  completed + incomplete == total: True
User 3: {'total': 20, 'completed': 7, 'incomplete': 13}  ->  completed + incomplete == total: True
User 4: {'total': 20, 'completed': 6, 'incomplete': 14}  ->  completed + incomplete == total: True
User 5: {'total': 20, 'completed': 12, 'incomplete': 8}  ->  completed + incomplete == total: True
User 6: {'total': 20, 'completed': 6, 'incomplete': 14}  ->  completed + incomplete == total: True
User 7: {'total': 20, 'completed': 9, 'incomplete': 11}  ->  completed + incomplete == total: True
User 8: {'total': 20, 'completed': 11, 'incomplete': 9}  ->  completed + incomplete == total: True
User 9: {'total': 20, 'completed': 8, 'incomplete': 12}  ->  completed + incomplete == total: True
User 10: {'total': 20, 'completed': 12, 'incomplete': 8}  ->  completed + incomplete == total: True


**(c) Completion rate per user; highest, lowest, and overall average**

In [51]:
completion_rates = {
    uid: round(s["completed"] / s["total"] * 100, 1)
    for uid, s in todo_stats.items() if s["total"] > 0
}

if completion_rates:
    highest_uid = max(completion_rates, key=completion_rates.get)
    lowest_uid = min(completion_rates, key=completion_rates.get)
    overall_average = round(sum(completion_rates.values()) / len(completion_rates), 1)

    print(f"Highest completion rate -> User {highest_uid}: {completion_rates[highest_uid]}%")
    print(f"Lowest completion rate -> User {lowest_uid}: {completion_rates[lowest_uid]}%")
    print(f"Overall average completion rate: {overall_average}%")
else:
    print("No completion rate data available (fetch the API with internet access to populate this).")


Highest completion rate -> User 5: 60.0%
Lowest completion rate -> User 4: 30.0%
Overall average completion rate: 45.0%


**(d) Formatted, rank-sorted table: Rank | Name | Total Tasks | Completed | Completion Rate %**

In [52]:
try:
    users_response = requests.get(USERS_URL, timeout=10)
    users_for_lookup = users_response.json() if users_response.status_code == 200 else []
except Exception:
    users_for_lookup = []

user_name_lookup = {u["id"]: u["name"] for u in users_for_lookup}

ranked = sorted(completion_rates.items(), key=lambda kv: kv[1], reverse=True)

header = f"{'Rank':<6}{'Name':<20}{'Total Tasks':<14}{'Completed':<12}{'Completion Rate %'}"
print(header)
for rank, (uid, rate) in enumerate(ranked, start=1):
    name = user_name_lookup.get(uid, f"User {uid}")
    total = todo_stats[uid]["total"]
    completed = todo_stats[uid]["completed"]
    print(f"{rank:<6}{name:<20}{total:<14}{completed:<12}{rate}")


Rank  Name                Total Tasks   Completed   Completion Rate %
1     Chelsey Dietrich    20            12          60.0
2     Clementina DuBuque  20            12          60.0
3     Leanne Graham       20            11          55.0
4     Nicholas Runolfsdottir V20            11          55.0
5     Kurtis Weissnat     20            9           45.0
6     Ervin Howell        20            8           40.0
7     Glenna Reichert     20            8           40.0
8     Clementine Bauch    20            7           35.0
9     Patricia Lebsack    20            6           30.0
10    Mrs. Dennis Schulist20            6           30.0


---
# Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis
*(requires live internet access to `jsonplaceholder.typicode.com`)*

**(a) `flatten_user()` -> single-level dict with the 10 required keys; apply to all users**

In [53]:
def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"],
    }

flattened_users = [flatten_user(u) for u in users_for_lookup] if users_for_lookup else []
for u in flattened_users:
    print(u)


{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'lat': '-37.3159', 'lng': '81.1496', 'company_name': 'Romaguera-Crona', 'company_catchphrase': 'Multi-layered client-server neural-net'}
{'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'lat': '-43.9509', 'lng': '-34.4618', 'company_name': 'Deckow-Crist', 'company_catchphrase': 'Proactive didactic contingency'}
{'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'city': 'McKenziehaven', 'zipcode': '59590-4157', 'lat': '-68.6102', 'lng': '-47.0653', 'company_name': 'Romaguera-Jacobson', 'company_catchphrase': 'Face to face bifurcated interface'}
{'id': 4, 'name': 'Patricia Lebsack', 'username': 'Karianne', 'email': 'Julianne.OConner@kory.org', 'city': 'South Elvis', 'zipcode': '53919-4257', 'lat': '29.4572', 'lng': '-164.2990', '

**(b) DataFrame from flattened users; cast lat/lng to float; add `distance_from_origin`; furthest user**

In [54]:
if flattened_users:
    df_users = pd.DataFrame(flattened_users)
    df_users["lat"] = df_users["lat"].astype(float)
    df_users["lng"] = df_users["lng"].astype(float)
    df_users["distance_from_origin"] = df_users.apply(
        lambda row: math.sqrt(row["lat"] ** 2 + row["lng"] ** 2), axis=1
    )

    print(df_users.to_string())

    furthest_user = df_users.loc[df_users["distance_from_origin"].idxmax()]
    print("\nUser with the greatest distance from origin:")
    print(furthest_user)
else:
    df_users = pd.DataFrame()
    print("No user data available (fetch the API with internet access to populate this).")


   id                      name          username                      email            city     zipcode      lat       lng        company_name                       company_catchphrase  distance_from_origin
0   1             Leanne Graham              Bret          Sincere@april.biz     Gwenborough  92998-3874 -37.3159   81.1496     Romaguera-Crona    Multi-layered client-server neural-net             89.318161
1   2              Ervin Howell         Antonette          Shanna@melissa.tv     Wisokyburgh  90566-7771 -43.9509  -34.4618        Deckow-Crist            Proactive didactic contingency             55.850669
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   McKenziehaven  59590-4157 -68.6102  -47.0653  Romaguera-Jacobson         Face to face bifurcated interface             83.201575
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org     South Elvis  53919-4257  29.4572 -164.2990       Robel-Corkery  Multi-tiered zero toler

**(c) Northern vs. Southern hemisphere groups by `lat`; hemisphere with higher average distance**

In [55]:
if not df_users.empty:
    northern = df_users[df_users["lat"] > 0]
    southern = df_users[df_users["lat"] < 0]

    print("Northern Hemisphere users:", list(northern["name"]))
    print("Southern Hemisphere users:", list(southern["name"]))

    north_avg = northern["distance_from_origin"].mean() if not northern.empty else 0
    south_avg = southern["distance_from_origin"].mean() if not southern.empty else 0

    winner = "Northern" if north_avg > south_avg else "Southern"
    print(f"\nNorthern avg distance: {north_avg:.2f} | Southern avg distance: {south_avg:.2f}")
    print(f"Hemisphere with the higher average distance from origin: {winner}")
else:
    print("No user data available (fetch the API with internet access to populate this).")


Northern Hemisphere users: ['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']
Southern Hemisphere users: ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']

Northern avg distance: 123.58 | Southern avg distance: 84.32
Hemisphere with the higher average distance from origin: Northern


**(d) Comment counts per post; top 5 and bottom 5 most/least-commented posts**

In [56]:
COMMENTS_URL = "https://jsonplaceholder.typicode.com/comments"

try:
    posts_all = fetch_json(POSTS_URL)
    comments_all = fetch_json(COMMENTS_URL)
except Exception as e:
    print("Could not reach the API from this environment:", e)
    posts_all, comments_all = [], []

if posts_all and comments_all:
    comment_counts = Counter(c["postId"] for c in comments_all)

    post_comment_summary = sorted(
        (
            (p["id"], p["title"][:40], comment_counts.get(p["id"], 0))
            for p in posts_all
        ),
        key=lambda t: t[2],
        reverse=True,
    )

    print("Top 5 most-commented posts:")
    for post_id, title, count in post_comment_summary[:5]:
        print(f"  Post {post_id}: '{title}' -> {count} comments")

    print("\nBottom 5 least-commented posts:")
    for post_id, title, count in post_comment_summary[-5:]:
        print(f"  Post {post_id}: '{title}' -> {count} comments")
else:
    print("No post/comment data available (fetch the API with internet access to populate this).")


Top 5 most-commented posts:
  Post 1: 'sunt aut facere repellat provident occae' -> 5 comments
  Post 2: 'qui est esse' -> 5 comments
  Post 3: 'ea molestias quasi exercitationem repell' -> 5 comments
  Post 4: 'eum et est occaecati' -> 5 comments
  Post 5: 'nesciunt quas odio' -> 5 comments

Bottom 5 least-commented posts:
  Post 96: 'quaerat velit veniam amet cupiditate aut' -> 5 comments
  Post 97: 'quas fugiat ut perspiciatis vero provide' -> 5 comments
  Post 98: 'laboriosam dolor voluptates' -> 5 comments
  Post 99: 'temporibus sit alias delectus eligendi p' -> 5 comments
  Post 100: 'at nam consequatur ea labore ea harum' -> 5 comments
